# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The demo client below is deterministic so the notebook can be executed without a model account. Replace it with an adapter implementing `generate(...)` for real extraction.

In [1]:
from pathlib import Path
import json
import re
from urllib.request import Request, urlopen

import networkx as nx

from semantic_graphicalizer import SemanticGraphicalizer

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
GUTENBERG_URL = 'https://www.gutenberg.org/cache/epub/53103/pg53103.txt'
GUTENBERG_PATH = RAW_DIR / 'pg53103.txt'

def download_gutenberg_text(refresh=False):
    if refresh or not GUTENBERG_PATH.exists():
        request = Request(GUTENBERG_URL, headers={'User-Agent': 'semantic-graphicalizer/0.1'})
        with urlopen(request, timeout=30) as response:
            GUTENBERG_PATH.write_bytes(response.read())
    return GUTENBERG_PATH.read_text(encoding='utf-8')

book_text = download_gutenberg_text()
len(book_text)

32588

In [2]:
def extract_tales(text, limit=5):
    body = text.split('*** START OF THE PROJECT GUTENBERG EBOOK', 1)[-1]
    body = body.split('*** END OF THE PROJECT GUTENBERG EBOOK', 1)[0]
    headings = list(re.finditer(r"(?m)^\s{5,}([A-ZÆŒ][A-ZÆŒ'’& ,.-]{2,80})\s*$", body))
    tales = []
    for index, heading in enumerate(headings):
        title = heading.group(1).strip()
        if len(title.split()) < 2:
            continue
        end = headings[index + 1].start() if index + 1 < len(headings) else len(body)
        tale = body[heading.start():end].strip()
        if len(tale) >= 250:
            tales.append(tale)
        if len(tales) == limit:
            break
    return tales

tales = extract_tales(book_text)
[(t.splitlines()[0], len(t)) for t in tales]

[('THE DAW IN BORROWED FEATHERS', 859),
 ('THE SUN AND THE WIND', 972),
 ('THE DOG IN THE MANGER', 304),
 ('MERCURY AND THE WOODMAN', 1654),
 ('THE FOX AND THE STORK', 1421)]

## Model adapter

For a real run, replace `DemoModel` with a provider-specific adapter. The core library only requires a `generate(stage=..., prompt=..., schema=..., context=...)` method returning a parsed mapping.

In [3]:
class DemoModel:
    def generate(self, *, stage, prompt, schema, context):
        if stage == 'summarize':
            return {'summary': 'A fable describes characters whose actions lead to a consequence and a moral.'}
        if stage == 'normalize':
            return {'normalized': 'A Character performs an Action that causes a consequence and teaches a Moral.'}
        if stage == 'decompose':
            return {'propositions': [{'id': 'p1', 'text': 'A character performs an action.', 'source_text': 'A fable action.'}, {'id': 'p2', 'text': 'The action teaches a moral.', 'source_text': 'A fable moral.'}]}
        if stage == 'triple':
            prefix = context['chunk_id']
            return {'triples': [
                {'id': 't1', 'proposition_id': f'{prefix}:p1', 'subject': {'mention': 'main character', 'label': 'Character'}, 'predicate': 'performs', 'object': {'mention': 'central action', 'label': 'Action'}, 'proposition': 'A character performs an action.'},
                {'id': 't2', 'proposition_id': f'{prefix}:p2', 'subject': {'mention': 'central action', 'label': 'Action'}, 'predicate': 'teaches', 'object': {'mention': 'moral lesson', 'label': 'Moral'}, 'proposition': 'The action teaches a moral.'},
            ]}
        raise ValueError(stage)

graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
    model=DemoModel(),
)
traces = graphicalizer.fit([tales[0]]).transform_with_trace([tales[0]])
trace = traces[0]
[(p.text, p.proposition_id) for p in trace.propositions]

[('A character performs an action.', 'document-0:chunk-0:p1'),
 ('The action teaches a moral.', 'document-0:chunk-0:p2')]

In [4]:
graph = trace.graph
print('nodes:', graph.number_of_nodes(), 'edges:', graph.number_of_edges())
print('node labels:', nx.get_node_attributes(graph, 'label'))
print('edge labels:', [data['label'] for _, _, data in graph.edges(data=True)])

nodes: 3 edges: 2
node labels: {'character::main_character': 'Character', 'action::central_action': 'Action', 'moral::moral_lesson': 'Moral'}
edge labels: ['A character performs an action.', 'The action teaches a moral.']
